# Modulprojekt DLT – BERT-Finetuning: Sentiment-Klassifikation

**Kurs:** Deep Learning Techniken für die CL
**Aufgabe:** Feintunen eines BERT-Modells für eine klassische NLP-Aufgabe.

Dieses Notebook ist eine **Vorlage**. Es implementiert die komplette Pipeline auf dem
Datensatz `rotten_tomatoes` (binäre Sentiment-Klassifikation). Ihre Aufgabe ist es,
die Pipeline zu verstehen, sauber auszuwerten und **eine eigene Untersuchung** hinzuzufügen
(siehe Abschnitt 8).

**Laufzeit:** Läuft auf der kostenlosen Colab-Stufe (T4-GPU) in wenigen Minuten,
auf CPU in ca. 10–30 Minuten. Setzen Sie in Colab: *Laufzeit → Laufzeittyp ändern → T4 GPU*.

**Wichtig für Reproduzierbarkeit:** Alle Zufallszahlen sind mit einem festen Seed
initialisiert. Das Test-Set wird **nur einmal ganz am Ende** benutzt.


## 0 – Einrichtung

Wir installieren die benötigten Pakete. In Colab müssen `transformers`, `datasets`
und `evaluate` in der Regel installiert werden; `peft` nur für die optionale LoRA-Aufgabe.


In [ ]:
# In Colab ausführen. Lokal ggf. in einer venv installieren.
%pip install -q "transformers>=4.40" "datasets>=2.19" evaluate scikit-learn peft accelerate
# In Colab ist ein zu altes torchao vorinstalliert, das mit aktuellem peft
# kollidiert (ImportError in Abschnitt 7). LoRA auf BERT braucht torchao nicht,
# daher entfernen wir es. Danach: Laufzeit NICHT neu starten, einfach weiter.
%pip uninstall -y -q torchao

In [ ]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 1 – Datensatz laden und inspizieren

**Bevor** man modelliert, schaut man sich die Daten an: Wie viele Beispiele gibt es?
Sind die Klassen balanciert? Wie lang sind die Texte? Das bestimmt spätere
Design-Entscheidungen (z. B. `max_length`).


In [ ]:
from datasets import load_dataset

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")   # Splits: train / validation / test
print(ds)
labels = ds["train"].features["label"].names   # ['neg', 'pos']
print("Labels:", labels)
ds["train"][0]

In [ ]:
import collections
# Klassenverteilung pro Split
for split in ds:
    c = collections.Counter(ds[split]["label"])
    print(f"{split:11s}: n={len(ds[split]):5d}  neg={c[0]:5d}  pos={c[1]:5d}")

In [ ]:
import matplotlib.pyplot as plt
# Längenverteilung (in Wörtern) – hilft beim Wählen von max_length
lengths = [len(t.split()) for t in ds["train"]["text"]]
plt.figure(figsize=(6,3))
plt.hist(lengths, bins=40)
plt.xlabel("Länge (Wörter)"); plt.ylabel("Anzahl"); plt.title("Textlängen (train)")
plt.tight_layout(); plt.show()
print("Median:", int(np.median(lengths)), " 95. Perzentil:", int(np.percentile(lengths,95)))

## 2 – Baselines

Ein Modell ist nur so gut wie der Vergleich, gegen den es antritt. Wir berechnen zwei
Baselines:

1. **Majority-Class** – rät immer die häufigste Klasse. Untergrenze für "sinnvoll".
2. **TF-IDF + logistische Regression** – ein starker, klassischer Vektorraum-Baseline
   (Anknüpfung an das Kapitel zu Vektorraum-Repräsentationen).

Ihr feingetuntes BERT sollte **beide** schlagen.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

Xtr, ytr = ds["train"]["text"], ds["train"]["label"]
Xva, yva = ds["validation"]["text"], ds["validation"]["label"]

# 1) Majority class
maj = collections.Counter(ytr).most_common(1)[0][0]
pred_maj = [maj]*len(yva)
print(f"Majority : acc={accuracy_score(yva,pred_maj):.3f}  macro-F1={f1_score(yva,pred_maj,average='macro'):.3f}")

# 2) TF-IDF + LogReg
vec = TfidfVectorizer(ngram_range=(1,2), min_df=2)
clf = LogisticRegression(max_iter=1000)
clf.fit(vec.fit_transform(Xtr), ytr)
pred_lr = clf.predict(vec.transform(Xva))
print(f"TF-IDF+LR: acc={accuracy_score(yva,pred_lr):.3f}  macro-F1={f1_score(yva,pred_lr,average='macro'):.3f}")

## 3 – Tokenisierung

Wir laden den Tokenizer zum gewählten Modell und wandeln die Texte in Token-IDs um.
`distilbert-base-uncased` ist klein und schnell – ideal ohne starke GPU. Sie können
später `bert-base-uncased` ausprobieren und vergleichen (Abschnitt 8).


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128   # aus der Längenverteilung oben abgeleitet
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

ds_tok = ds.map(tokenize, batched=True)
ds_tok = ds_tok.remove_columns(["text"]).rename_column("label", "labels")
# Hinweis: KEIN set_format("torch") -- in Colab loest der Torch-Formatter von
# datasets teils einen ImportError (torchvision VideoReader) aus. Der
# DataCollatorWithPadding wandelt die Batches ohnehin selbst in Tensoren um.
ds_tok["train"][0].keys()

## 4 – Feintuning mit dem `Trainer`

Wir laden `AutoModelForSequenceClassification` mit einem frischen Klassifikationskopf
(2 Labels) und trainieren mit der `Trainer`-API aus Kapitel 3 des HF-Kurses.

`compute_metrics` berechnet **Accuracy und Macro-F1**. Bei balancierten Daten sind beide
ähnlich; bei unbalancierten Aufgaben ist Macro-F1 aussagekräftiger.


In [ ]:
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0:"neg",1:"pos"}, label2id={"neg":0,"pos":1})

collator = DataCollatorWithPadding(tokenizer)  # dynamisches Padding pro Batch

def compute_metrics(eval_pred):
    logits, y = eval_pred
    pred = logits.argmax(-1)
    return {"accuracy": accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro")}

args = TrainingArguments(
    output_dir="out",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer.train()

## 5 – Evaluation auf dem Test-Set

Jetzt – und erst jetzt – benutzen wir das Test-Set. Wir berichten Accuracy und
Macro-F1, den vollständigen `classification_report` und eine Confusion-Matrix.


In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

pred_out = trainer.predict(ds_tok["test"])
y_true = pred_out.label_ids
y_pred = pred_out.predictions.argmax(-1)

print(f"BERT test: acc={accuracy_score(y_true,y_pred):.3f}  "
      f"macro-F1={f1_score(y_true,y_pred,average='macro'):.3f}\n")
print(classification_report(y_true, y_pred, target_names=labels))

ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=labels)
plt.title("Confusion-Matrix (Test)"); plt.tight_layout(); plt.show()

## 6 – Qualitative Fehleranalyse

Zahlen allein sagen wenig. Schauen Sie sich falsch klassifizierte Beispiele an und
bilden Sie **Hypothesen**: Was verwirrt das Modell? Negationen? Ironie? Lange Sätze?
Diese Beobachtungen gehören in Ihren Bericht.


In [ ]:
test_texts = ds["test"]["text"]
errors = [(test_texts[i], labels[y_true[i]], labels[y_pred[i]])
          for i in range(len(y_true)) if y_true[i] != y_pred[i]]
print(f"{len(errors)} Fehler von {len(y_true)} Beispielen\n")
for txt, gold, pred in errors[:10]:
    print(f"[gold={gold} | pred={pred}] {txt[:120]}")

## 7 – (Optional) LoRA / Parameter-effizientes Feintuning

Statt alle Gewichte zu aktualisieren, trainiert **LoRA** nur kleine Adapter-Matrizen.
Bei BERT-base ist volles Feintuning schon billig – LoRA ist hier vor allem eine
**Untersuchung**: Wie viele Parameter werden trainiert, und wie ändert sich die
Genauigkeit? Vergleichen Sie mit Abschnitt 4.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=["q_lin","v_lin"],   # DistilBERT-Attention; bei BERT: ["query","value"]
)
lora_model = get_peft_model(base, lora_cfg)
lora_model.print_trainable_parameters()   # vergleiche mit ~67M des vollen Modells

# Danach: denselben Trainer-Block wie in Abschnitt 4, aber model=lora_model.

## 8 – (Pflicht) Ihre eigene Untersuchung

Wählen Sie **mindestens eine** Erweiterung und werten Sie sie sauber aus:

- **Daten-Effizienz / Lernkurve:** Accuracy vs. Trainingsgröße (z. B. 250 / 1000 / 5000).
- **Volles Feintuning vs. LoRA vs. eingefrorener Encoder** (nur linearer Kopf).
- **Modellvergleich:** DistilBERT vs. BERT-base vs. `prajjwal1/bert-mini`
  (Accuracy / Größe / Trainingszeit).
- **Robustheit:** Wie fällt die Leistung auf gestörten Eingaben (Tippfehler, Groß-/Kleinschreibung)?
- **Hyperparameter-Sensitivität:** Lernrate / Epochen bei festem Rechenbudget.

Der Code unten ist ein Startpunkt für die **Lernkurve**.


In [ ]:
# Vollständiges Beispiel: Lernkurve (Macro-F1 vs. Trainingsgröße).
# Wir kapseln Training + Evaluation in eine Funktion und rufen sie in einer Schleife.
# Für Robustheit sollte man je Größe mehrere Seeds mitteln; hier ein Seed pro Größe,
# damit es auf CPU schnell bleibt.

def train_eval(n_train, seed=SEED, epochs=3):
    """Trainiert DistilBERT auf n_train Beispielen und gibt (acc, macro_f1) auf
    dem validation-Split zurück. Nutzt dieselbe Tokenisierung wie oben."""
    torch.manual_seed(seed)
    sub = ds_tok["train"].shuffle(seed=seed).select(range(n_train))

    m = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2,
        id2label={0: "neg", 1: "pos"}, label2id={"neg": 0, "pos": 1})

    a = TrainingArguments(
        output_dir=f"out_lc_{n_train}",
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy="no",       # wir evaluieren einmal am Ende selbst
        save_strategy="no",
        seed=seed,
        report_to="none",
        logging_strategy="no",
        disable_tqdm=True,
    )
    tr = Trainer(model=m, args=a, train_dataset=sub,
                 data_collator=collator, compute_metrics=compute_metrics)
    tr.train()

    out = tr.predict(ds_tok["validation"])
    yp = out.predictions.argmax(-1)
    yt = out.label_ids
    return accuracy_score(yt, yp), f1_score(yt, yp, average="macro")


sizes = [250, 1000, 5000]
lc = {}
for n in sizes:
    acc, mf1 = train_eval(n)
    lc[n] = {"accuracy": acc, "macro_f1": mf1}
    print(f"n={n:5d}  acc={acc:.3f}  macro-F1={mf1:.3f}")

# Plot der Lernkurve
plt.figure(figsize=(6, 3))
plt.plot(sizes, [lc[n]["macro_f1"] for n in sizes], marker="o")
plt.xscale("log")
plt.xlabel("Trainingsgröße (log)"); plt.ylabel("Macro-F1 (validation)")
plt.title("Lernkurve"); plt.grid(True, alpha=.3); plt.tight_layout(); plt.show()

# Ergebnisse für den Bericht sichern
import json, os
os.makedirs("results", exist_ok=True)
with open("results/learning_curve.json", "w") as f:
    json.dump(lc, f, indent=2)


## 9 – Speichern der Ergebnisse

Speichern Sie Modell und Metriken reproduzierbar. Committen Sie die `metrics.json`
(nicht das große Modell) ins Repository.


In [ ]:
import json, os
os.makedirs("results", exist_ok=True)
metrics = {
    "model": MODEL_NAME, "seed": SEED, "max_length": MAX_LENGTH,
    "test_accuracy": float(accuracy_score(y_true, y_pred)),
    "test_macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    "baseline_majority_f1": float(f1_score(yva, pred_maj, average="macro")),
    "baseline_tfidf_f1": float(f1_score(yva, pred_lr, average="macro")),
}
with open("results/metrics.json","w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))
# trainer.save_model("results/model")   # optional, groß